In [0]:
# ============================================================
# ÉTAPE 11-12 — Configuration de l'accès ADLS (Serverless)
# ============================================================
storage_account_name = "retailstraccount"
container_name       = "retailcontainer"
storage_account_key  = ""

azure_key_conf = f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net"
base_path      = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"

print("✔ Variables configurées.")
print(f"Base path : {base_path}")

# COMMAND ----------

# ============================================================
# ÉTAPE 13 — Chargement des données Bronze
# ============================================================
bronze_path = f"{base_path}/bronze"

try:
    df_transactions = (spark.read
        .format("parquet")
        .option(azure_key_conf, storage_account_key)
        .load(f"{bronze_path}/transaction/"))
    print("✔ Transactions chargées.")

    df_products = (spark.read
        .format("parquet")
        .option(azure_key_conf, storage_account_key)
        .load(f"{bronze_path}/product/"))
    print("✔ Products chargés.")

    df_stores = (spark.read
        .format("parquet")
        .option(azure_key_conf, storage_account_key)
        .load(f"{bronze_path}/store/"))
    print("✔ Stores chargés.")

    df_customers = (spark.read
        .format("parquet")
        .option(azure_key_conf, storage_account_key)
        .load(f"{bronze_path}/customer/"))
    print("✔ Customers chargés.")

    print("\n[SUCCÈS] Les 4 DataFrames Bronze sont prêts !")
    df_transactions.printSchema()
    display(df_customers.limit(5))

except Exception as e:
    print("[ERREUR] :", e)

# COMMAND ----------

print("=== TRANSACTIONS ===")
df_transactions.printSchema()
df_transactions.show(3)

print("=== PRODUCTS ===")
df_products.printSchema()
df_products.show(3)

print("=== STORES ===")
df_stores.printSchema()
df_stores.show(3)

print("=== CUSTOMERS ===")
df_customers.printSchema()
df_customers.show(3)

# COMMAND ----------

# ============================================================
# ÉTAPE 14 — Nettoyage des données
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType

# --- Transactions ---
df_transactions_clean = df_transactions \
    .withColumn("transaction_id",   F.col("transaction_id").cast(IntegerType())) \
    .withColumn("product_id",       F.col("product_id").cast(IntegerType())) \
    .withColumn("store_id",         F.col("store_id").cast(IntegerType())) \
    .withColumn("customer_id",      F.col("customer_id").cast(IntegerType())) \
    .withColumn("quantity",         F.col("quantity").cast(IntegerType())) \
    .withColumn("transaction_date", F.to_date(F.col("transaction_date")))

df_transactions_clean = df_transactions_clean.dropDuplicates(["transaction_id"])
df_transactions_clean = df_transactions_clean.dropna(
    subset=["transaction_id", "product_id", "store_id", "quantity"]
)
df_transactions_clean = df_transactions_clean.filter(F.col("quantity") > 0)

print("Transactions après nettoyage :", df_transactions_clean.count())
df_transactions_clean.show(3)

# COMMAND ----------

# --- Products ---
df_products_clean = df_products \
    .withColumn("product_id",   F.col("product_id").cast(IntegerType())) \
    .withColumn("price",        F.col("price").cast(FloatType())) \
    .withColumn("product_name", F.trim(F.col("product_name"))) \
    .withColumn("category",     F.trim(F.col("category")))

df_products_clean = df_products_clean.dropDuplicates(["product_id"])
df_products_clean = df_products_clean.dropna(subset=["product_id", "product_name"])

print("Products après nettoyage :", df_products_clean.count())
df_products_clean.show(3)

# COMMAND ----------

# --- Stores ---
df_stores_clean = df_stores \
    .withColumn("store_id",   F.col("store_id").cast(IntegerType())) \
    .withColumn("store_name", F.trim(F.col("store_name"))) \
    .withColumn("location",   F.trim(F.col("location")))

df_stores_clean = df_stores_clean.dropDuplicates(["store_id"])
df_stores_clean = df_stores_clean.dropna(subset=["store_id", "store_name"])

print("Stores après nettoyage :", df_stores_clean.count())
df_stores_clean.show(3)

# COMMAND ----------

# --- Customers ---
df_customers_clean = df_customers \
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType())) \
    .withColumn("first_name",  F.trim(F.col("first_name"))) \
    .withColumn("last_name",   F.trim(F.col("last_name"))) \
    .withColumn("email",       F.lower(F.trim(F.col("email")))) \
    .dropna(subset=["customer_id"]) \
    .dropDuplicates(["customer_id"])

print("Customers après nettoyage :", df_customers_clean.count())
df_customers_clean.show(3)

# COMMAND ----------

# ============================================================
# ÉTAPES 15-16 — Jointures + total_amount
# ============================================================
df_silver = df_transactions_clean \
    .join(df_products_clean,  on="product_id",  how="left") \
    .join(df_stores_clean,    on="store_id",    how="left") \
    .join(df_customers_clean, on="customer_id", how="left")

df_silver = df_silver.withColumn(
    "total_amount",
    F.round(F.col("quantity") * F.col("price"), 2)
)

df_silver = df_silver.select(
    "transaction_id",
    "transaction_date",
    "customer_id",
    F.concat_ws(" ", F.col("first_name"), F.col("last_name")).alias("customer_name"),
    "email",
    "product_id",
    "product_name",
    "category",
    "store_id",
    "store_name",
    "location",
    "quantity",
    "price",
    "total_amount"
)

print("Silver — lignes :", df_silver.count())
df_silver.show(5)

# COMMAND ----------

# ============================================================
# ÉTAPE 17 — Sauvegarde Silver au format Delta
# ============================================================
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option(azure_key_conf, storage_account_key)
    .save(f"{base_path}/silver/sales_silver"))

print("✔ Silver sauvegardé en Delta dans ADLS.")

# COMMAND ----------

# ============================================================
# ÉTAPE 18 — Lecture Silver
# ============================================================
df_silver = (spark.read
    .format("delta")
    .option(azure_key_conf, storage_account_key)
    .load(f"{base_path}/silver/sales_silver"))

print("✔ Silver chargé — lignes :", df_silver.count())
df_silver.show(5)

# COMMAND ----------

# ============================================================
# ÉTAPE 19 — Agrégations Gold
# ============================================================

# Par produit
df_gold_product = df_silver.groupBy("product_id", "product_name", "category") \
    .agg(
        F.sum("quantity").alias("total_quantity_sold"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.count("transaction_id").alias("nb_transactions"),
        F.round(F.avg("total_amount"), 2).alias("avg_transaction_value")
    )

# Par date
df_gold_date = df_silver.groupBy("transaction_date") \
    .agg(
        F.sum("quantity").alias("total_quantity_sold"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.count("transaction_id").alias("nb_transactions"),
        F.round(F.avg("total_amount"), 2).alias("avg_transaction_value")
    ).orderBy("transaction_date")

# Par magasin
df_gold_store = df_silver.groupBy("store_id", "store_name", "location") \
    .agg(
        F.sum("quantity").alias("total_quantity_sold"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.count("transaction_id").alias("nb_transactions"),
        F.round(F.avg("total_amount"), 2).alias("avg_transaction_value")
    )

print("✔ Agrégations Gold calculées.")
df_gold_product.show(5)
df_gold_date.show(5)
df_gold_store.show(5)

# COMMAND ----------

# ============================================================
# ÉTAPE 20 — Sauvegarde Gold au format Delta
# ============================================================
gold_path = f"{base_path}/gold"

(df_gold_product.write
    .format("delta")
    .mode("overwrite")
    .option(azure_key_conf, storage_account_key)
    .save(f"{gold_path}/sales_by_product"))

(df_gold_date.write
    .format("delta")
    .mode("overwrite")
    .option(azure_key_conf, storage_account_key)
    .save(f"{gold_path}/sales_by_date"))

(df_gold_store.write
    .format("delta")
    .mode("overwrite")
    .option(azure_key_conf, storage_account_key)
    .save(f"{gold_path}/sales_by_store"))

# ============================================================
# ÉTAPE 21 — Export Gold en CSV
# ============================================================

# Par produit
(df_gold_product.coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header", "true")
    .option(azure_key_conf, storage_account_key)
    .save(f"{base_path}/gold/csv/sales_by_product"))

print("✔ sales_by_product exporté en CSV.")

# Par date
(df_gold_date.coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header", "true")
    .option(azure_key_conf, storage_account_key)
    .save(f"{base_path}/gold/csv/sales_by_date"))

print("✔ sales_by_date exporté en CSV.")

# Par magasin
(df_gold_store.coalesce(1)
    .write
    .format("csv")
    .mode("overwrite")
    .option("header", "true")
    .option(azure_key_conf, storage_account_key)
    .save(f"{base_path}/gold/csv/sales_by_store"))

print("✔ sales_by_store exporté en CSV.")
print("\n[SUCCÈS] Les 3 fichiers CSV Gold sont disponibles dans ADLS.")

print("✔ Gold sauvegardé en Delta dans ADLS.")

✔ Variables configurées.
Base path : abfss://retailcontainer@retailstraccount.dfs.core.windows.net
✔ Transactions chargées.
✔ Products chargés.
✔ Stores chargés.
✔ Customers chargés.

[SUCCÈS] Les 4 DataFrames Bronze sont prêts !
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: date (nullable = true)



customer_id,first_name,last_name,email,phone,city,registration_date
101,Youssef,Alaoui,user101@example.com,0612345678,Casablanca,2023-09-14
102,Salma,Bennani,user102@example.com,0623456789,Rabat,2024-01-21
103,Omar,El Idrissi,user103@example.com,0634567890,Marrakech,2023-07-10
104,Imane,Tazi,user104@example.com,0645678901,Fès,2024-02-05
105,Hamza,Amrani,user105@example.com,0656789012,Tanger,2023-06-28


=== TRANSACTIONS ===
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: date (nullable = true)

+--------------+-----------+----------+--------+--------+----------------+
|transaction_id|customer_id|product_id|store_id|quantity|transaction_date|
+--------------+-----------+----------+--------+--------+----------------+
|             1|        227|         8|       4|       4|      2025-03-31|
|             2|        205|         3|       4|       5|      2024-11-12|
|             3|        216|         2|       2|       3|      2025-05-01|
+--------------+-----------+----------+--------+--------+----------------+
only showing top 3 rows
=== PRODUCTS ===
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: d